# Local models

## Ollama

**make sure you've 1. installed ollama, downloaded (pulled) the model, and started "ollama serve" as explained in the book!**

In [1]:
from langchain_ollama import ChatOllama

chat = ChatOllama(
    model="deepseek-r1:1.5b",
    temperature=0,
)

messages = [
    (
        "system",
        "You are a helpful assistant.",
    ),
    ("human", "What makes LangChain great for working with LLMs?"),
]
ai_msg = chat.invoke(messages)
print(ai_msg.content)

<think>
Okay, so I'm trying to understand why LangChain is considered great for working with LLMs. From what I know, LangChain is an open-source project that allows developers to create custom chatbots using large language models like GPT-3. It's built on PyTorch and provides a lot of flexibility.

First, I think about the main features of LangChain. There are components like ChatGPT, LLMs, and interfaces for building chatbots. These components make it easier to integrate different parts of the system together. For example, you can use an LLM as part of your chatbot or build a custom interface from scratch.

I also remember that LangChain supports various programming languages like Python, JavaScript, and Ruby. This means developers can write their own modules or extend existing ones, which is really useful for customization. It's not just about using pre-built components; it's about creating something unique.

Another point is the flexibility in how chatbots are built. You can choose 

## Huggingface

In [3]:
# Huggingface is another way to run models locally

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

# Create a pipeline with a small model:
llm = HuggingFacePipeline.from_model_id(
    model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.03,
    ),
)

chat_model = ChatHuggingFace(llm=llm)

# Use it like any other LangChain LLM
messages = [
    SystemMessage(content="You're a helpful assistant"),
    HumanMessage(
        content="Explain the concept of machine learning in simple terms"
    ),
]
ai_msg = chat_model.invoke(messages)
print(ai_msg.content)

tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\work\code\generative_ai_with_langchain\.venv\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mpapa\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cpu


<|system|>
You're a helpful assistant</s>
<|user|>
Explain the concept of machine learning in simple terms</s>
<|assistant|>
Machine learning is a field that involves the use of algorithms and data to learn from and improve upon the behavior of machines or systems. It is a powerful tool for automating tasks and improving the performance of systems by learning from data and making predictions based on that data.

In simple terms, machine learning is the process of using algorithms to analyze and learn from data. This process involves collecting large amounts of data, training the algorithm to identify patterns and relationships in that data, and then using that knowledge to make predictions or make decisions.

For example, a machine learning algorithm might be used to predict the likelihood of a customer purchasing a particular product based on their previous purchase history and other factors. Or it might be used to recommend products to a user based on their browsing and purchase hist

In [ ]:
# integrating Ollama with LCEL patterns
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# initialize Ollama with the selected model
local_llm = ChatOllama(
    model="deepseek-r1:1.5b",
    temperature=0,
)

# create an LCEL chain using the local model
prompt = PromptTemplate.from_template("Explain {concept} in simple terms")
local_chain = prompt | local_llm | StrOutputParser()

# use the chain with your local model
result = local_chain.invoke({"concept": "quantum computing"})   
print(result)


In [7]:
# Resource management when working with local models
# configure model with optimized memory and processing settings
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
import time

def safe_model_call(llm, prompt, max_retries=2):
    retries = 0
    while retries < max_retries:
        try:
            return llm.invoke(prompt)
        except RuntimeError as e:
            # common error with local models when running out of VRAM
            if "CUDA out of memory" in str(e):
                print(f"GPU memory error, waiting and retrying ({retries+1}/{max_retries+1})...")
                time.sleep(2)  # give system time to free resources
                retries += 1
            else:
                print(f"RuntimeError: {e}")
                return "An error occurred while processing your request."
        except Exception as e:
            print(f"Unexpected error while calling model: {e}")
            return "An error occurred while processing your request."
        
    # if we exhausted retries
    return "Model is currently experiencing high load. Please try again later."


llm = ChatOllama(
    model="mistral:7b-instruct-q4_K_M", # 4-bit quantized model (smaller memory footprint)
    num_gpu=1, # 1 GPU - matches most desktops
    num_thread=8, # number of threads to use - usually matches physical cores
)

prompt = PromptTemplate.from_template("Explain {concept} in simple terms")
safe_llm = RunnableLambda(lambda x: safe_model_call(llm, x))
safe_chain = prompt | safe_llm | StrOutputParser()
response = safe_chain.invoke({"concept": "blockchain technology"})
print(response)


Blockchain technology is a type of digital record-keeping that allows information to be stored and transferred securely and transparently. Think of it like a digital ledger, where all transactions are recorded and verified by multiple parties before being added to the chain.

The decentralized nature of blockchain means that there is no single entity in control of the data, making it resistant to hacking and fraud. Each block in the chain contains a unique set of data, and once a block is added to the chain, it cannot be changed or deleted.

Blockchain technology is used in a variety of applications, including cryptocurrencies like Bitcoin, smart contracts, supply chain management, and identity verification.
